# 02 · Tools

In notebook 01 the model invented ShipFast's refund policy and an order status, because it cannot see
either. **Tools** fix that: functions the model can ask your code to run, whose results it then reads.

**You will learn**

- where the tools live, and why they are plain Python functions
- how Strands turns a function into a tool the model can see
- what a tool call looks like, and how Strands runs it for you
- how tool errors and safety checks work when the model gets something wrong

In [1]:
from strands import Agent

from agent_contracts.harness import verify_model
from strands_agents_impl.llm import make_model
from strands_agents_impl.prompts import SPECIALIST

verify_model()
model = make_model()

## The tools are plain functions

ShipFast's five tools live once, in `agent_contracts.shipfast.tools`, shared by all three tutorials. They
know nothing about Strands: each takes simple arguments and returns a JSON string. You can call them
directly.

In [2]:
import json

from agent_contracts.shipfast import tools as shipfast

shipfast.reset()  # clears the refund ledger, the tickets and the tool-call log
print([f.__name__ for f in shipfast.TOOLS])
print(json.dumps(json.loads(shipfast.lookup_order("A1001")), indent=2))

['lookup_order', 'track_shipment', 'search_policy', 'issue_refund', 'escalate_to_human']
{
  "order_id": "A1001",
  "customer": "Maria Santos",
  "item": "Wireless Headphones",
  "total": 89.0,
  "status": "in_transit",
  "carrier": "LBC",
  "ordered_on": "2026-10-28",
  "eta": "2026-11-04",
  "delivered_on": null
}


## Turning a function into a tool

Strands' `tool` reads a function's name, type hints and docstring and builds a **tool spec**: the name,
description and JSON input schema the model sees. `strands_agents_impl/tools.py` wraps all five:

In [3]:
import inspect

import strands_agents_impl.tools
from strands_agents_impl.tools import shipfast_tools

print(inspect.getsource(strands_agents_impl.tools.shipfast_tools))
tools = shipfast_tools()
for t in tools:
    print(f"{t.tool_name:18} {t.tool_spec['inputSchema']['json']['properties']}")

def shipfast_tools() -> list:
    return [tool(f) for f in shipfast.TOOLS]

lookup_order       {'order_id': {'description': 'Parameter order_id', 'type': 'string'}}
track_shipment     {'order_id': {'description': 'Parameter order_id', 'type': 'string'}}
search_policy      {'query': {'description': 'Parameter query', 'type': 'string'}}
issue_refund       {'order_id': {'description': 'Parameter order_id', 'type': 'string'}, 'amount': {'description': 'Parameter amount', 'type': 'number'}, 'reason': {'description': 'Parameter reason', 'type': 'string'}}
escalate_to_human  {'order_id': {'description': 'Parameter order_id', 'type': 'string'}, 'summary': {'description': 'Parameter summary', 'type': 'string'}}


The docstring is the only instruction the model gets about when to use a tool, so it matters as much
as the code. `issue_refund`'s docstring, for example, says the amount cannot exceed the order total.

## A tool call

Strands' `Agent` runs tools for you from the start, so to see a raw tool call we go one level down, to the
model itself. `model.stream(...)` sends one request with the tool specs and streams back the reply; a
tool call arrives as a `toolUse` block with the tool's name and its input. Nothing is run.

In [4]:
async def first_move(text: str) -> list[dict]:
    """The model's first reply to a message, as the tool calls it asks for (nothing is run)."""
    calls, current = [], None
    async for event in model.stream([{"role": "user", "content": [{"text": text}]}],
                                    tool_specs=[t.tool_spec for t in tools], system_prompt=SPECIALIST):
        start = event.get("contentBlockStart", {}).get("start", {})
        delta = event.get("contentBlockDelta", {}).get("delta", {})
        if "toolUse" in start:
            current = {"name": start["toolUse"]["name"], "input": ""}
            calls.append(current)
        elif "toolUse" in delta and current is not None:
            current["input"] += delta["toolUse"].get("input", "")
    return calls


print(await first_move("Hi, where is my order A1001?"))

[{'name': 'track_shipment', 'input': '{"order_id": "A1001"}'}]


## Letting the agent run it

Give the same tools to an `Agent` and it runs the whole loop: call the model, run the tool it asks for,
hand back the result, call the model again, until it answers in text. The conversation it builds shows
each step: a `toolUse` block from the model, then a `toolResult` block carrying the tool's output.

In [5]:
agent = Agent(model=model, tools=tools, system_prompt=SPECIALIST, callback_handler=None)
result = agent("Hi, where is my order A1001?")

for message in agent.messages:
    for block in message["content"]:
        if "toolUse" in block:
            print(f"{message['role']:9} toolUse    {block['toolUse']['name']} {block['toolUse']['input']}")
        elif "toolResult" in block:
            print(f"{message['role']:9} toolResult {block['toolResult']['content'][0]['text'][:90]} ...")
        elif "text" in block:
            print(f"{message['role']:9} text       {block['text'].strip()[:200]}")

user      text       Hi, where is my order A1001?
assistant toolUse    track_shipment {'order_id': 'A1001'}
user      toolResult {"order_id": "A1001", "status": "in_transit", "carrier": "LBC", "eta": "2026-11-04", "deli ...
assistant text       Your order A1001 is currently in transit with LBC. It is expected to arrive by November 4, 2026. The latest tracking events are:

- October 28, 2026: Order packed at Manila warehouse
- October 30, 202


That is the whole idea of an agent: in LangGraph and CrewAI notebook 02 you ran this loop by hand
first; in Strands the agent is the starting point. Notebook 03 looks at the loop more closely.

## Which tool does it reach for?

Here is the model's **first** move on the opening message of each ShipFast scenario, read from the stream
as above. Nothing is run.

In [6]:
from agent_contracts.shipfast.scoring import load_scenarios

seen = set()
for scenario in load_scenarios():
    text = scenario["turns"][0]
    if text in seen:
        continue
    seen.add(text)
    calls = await first_move(text)
    asked = ", ".join(f"{c['name']}({c['input']})" for c in calls) or "(no tool)"
    print(f"{text[:48]:50} -> {asked}")

Hi, where is my order A1001?                       -> track_shipment({"order_id": "A1001"})


My order A1002 never arrived and tracking says i   -> lookup_order({"order_id": "A1002"})


Order A1003 is more than a week late. Please can   -> lookup_order({"order_id": "A1003"})


My vase set from order A1004 arrived broken.       -> lookup_order({"order_id": "A1004"})


I want to return order A1005, the stand wobbles.   -> lookup_order({"order_id": "A1005"})


Where is order A9999?                              -> lookup_order({"order_id": "A9999"})


Can you write me a poem about cats?                -> (no tool)


Look for two things: whether each choice makes sense for the message, and the out-of-scope request,
where the right move is no tool at all.

The instructions' first step is "look up the order with lookup_order", and in the saved run Strands' model
followed it on five of the six in-scope messages. With the same instructions and model, the LangGraph and
CrewAI runs mostly opened with `track_shipment` instead. Each framework packages the system prompt, the
tool specs and the messages a little differently, and a small model notices.

## When a tool says no

Tools report problems as data, not exceptions, so the model can read them and recover. An unknown order
comes back as `order_not_found`, and policy P6 says to ask the customer to check the ID.

In [7]:
shipfast.reset()
result = Agent(model=model, tools=tools, system_prompt=SPECIALIST, callback_handler=None)("Where is order A9999?")
print("tools:", shipfast.tool_calls())
print("reply:", str(result).strip())

tools: ['lookup_order']
reply: The order ID A9999 could not be found. Please double-check the order ID and try again. If you need further assistance, feel free to ask!


## Safety lives in the tool, not the prompt

In the smoke test for this project, the model once tried to refund $120 on a $45 order without looking up
the total. The prompt cannot prevent that; the tool can. `issue_refund` refuses any amount above the
order total, or above what is left after earlier refunds, whoever calls it.

In [8]:
shipfast.reset()
print(shipfast.issue_refund("A1002", 120, "lost package"))
print(shipfast.issue_refund("A1002", 45, "lost package"))
print(shipfast.issue_refund("A1002", 45, "lost package, again"))
print("ledger:", shipfast.refunds())

{"error": "amount_exceeds_order_total", "order_id": "A1002", "amount": 120.0, "order_total": 45.0}
{"refund_id": "RF-A1002", "order_id": "A1002", "amount": 45.0, "status": "issued"}
{"error": "amount_exceeds_remaining", "order_id": "A1002", "amount": 45.0, "already_refunded": 45.0, "remaining": 0.0}
ledger: [{'refund_id': 'RF-A1002', 'order_id': 'A1002', 'amount': 45.0, 'status': 'issued', 'reason': 'lost package'}]


The ledger is what ShipFast's test runner trusts: a result only counts as `refunded` if the ledger
has a matching refund. The model's claim is never enough.

The shared tools also record every call, whichever framework made it. Notebook 07 uses this so all three
tutorials count tool calls the same way.

In [9]:
print(shipfast.tool_calls())
shipfast.reset()

['issue_refund', 'issue_refund', 'issue_refund']


## How the others do this

- **LangGraph** (LangChain) wraps the same functions with `tool()` and attaches them with
  `llm.bind_tools(tools)`; calling the model returns tool calls you can run by hand, and `create_agent`
  runs the loop.
- **CrewAI** wraps them with its own `tool` decorator; `LLM.call(..., tools=...)` returns raw tool calls,
  and an `Agent` runs the loop.

All three wrap the same five functions from `agent_contracts`, so the tools behave identically; only the
wrapping differs.

## Takeaways

- A tool is a plain function plus a spec the model reads; keep the logic outside the framework.
- The model only *asks* for a tool; in Strands the `Agent` runs it and hands back the result.
- `agent.messages` shows every `toolUse` and `toolResult`, so the loop is never hidden.
- Errors come back as data the model can act on, and safety checks belong in the tool.
- Next: **03 · Agent**.